In [ ]:
1

In [ ]:
import os
import librosa
import numpy as np

ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_PATH = os.path.join(ROOT, 'genres_stems')
GENRES = ["blues", "classical", "country", "disco", "hiphop", "jazz", "metal", "pop", "reggae", "rock"]

durations = []

for genre in GENRES:
    genre_path = os.path.join(STEMS_PATH, genre)
    for song in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song)
        if os.path.isdir(song_path) and genre == "jazz":
            for stem in os.listdir(song_path):
                if stem.endswith(".wav"):
                    file_path = os.path.join(song_path, stem)
                    try:
                        y, sr = librosa.load(file_path, sr=None)
                        durations.append(len(y) / sr)
                    except:
                        pass

print("Mean duration (Jazz stems):", np.mean(durations))


2

In [ ]:
sample_rates = set()

for genre in GENRES:
    genre_path = os.path.join(STEMS_PATH, genre)
    for song in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song)
        if os.path.isdir(song_path):
            for stem in os.listdir(song_path):
                if stem.endswith(".wav"):
                    file_path = os.path.join(song_path, stem)
                    try:
                        y, sr = librosa.load(file_path, sr=None)
                        sample_rates.add(sr)
                    except:
                        pass

print("Unique Sample Rates:", sorted(list(sample_rates)))


3

In [ ]:
corrupted = 0

for genre in GENRES:
    genre_path = os.path.join(STEMS_PATH, genre)
    for song in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song)
        if os.path.isdir(song_path):
            for stem in os.listdir(song_path):
                if stem.endswith(".wav"):
                    file_path = os.path.join(song_path, stem)
                    if os.path.getsize(file_path) == 0:
                        corrupted += 1
                    else:
                        try:
                            y, sr = librosa.load(file_path, sr=None)
                        except:
                            corrupted += 1

print("Corrupted or zero-byte files:", corrupted)


4

In [ ]:
peak_db = []

for genre in GENRES:
    genre_path = os.path.join(STEMS_PATH, genre)
    for song in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song)
        if os.path.isdir(song_path):
            vocal_file = os.path.join(song_path, "vocals.wav")
            if os.path.exists(vocal_file):
                try:
                    y, sr = librosa.load(vocal_file, sr=None)
                    peak = np.max(np.abs(y))
                    peak_db.append(20 * np.log10(peak + 1e-6))
                except:
                    pass

print("Average peak amplitude (dB):", np.mean(peak_db))


5

In [ ]:
centroids = []

for song in os.listdir(os.path.join(STEMS_PATH, "blues")):
    song_path = os.path.join(STEMS_PATH, "blues", song)
    if os.path.isdir(song_path):
        file_path = os.path.join(song_path, "other.wav")
        if os.path.exists(file_path):
            y, sr = librosa.load(file_path, sr=22050, duration=10)
            centroids.append(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))

print("Mean spectral centroid (Blues):", np.mean(centroids))


6

In [ ]:
genre_centroid = {}

for genre in GENRES:
    centroids = []
    genre_path = os.path.join(STEMS_PATH, genre)
    for song in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song)
        if os.path.isdir(song_path):
            file_path = os.path.join(song_path, "other.wav")
            if os.path.exists(file_path):
                y, sr = librosa.load(file_path, sr=22050, duration=10)
                centroids.append(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    genre_centroid[genre] = np.mean(centroids)

print("Genre with highest spectral centroid:", max(genre_centroid, key=genre_centroid.get))


7

In [ ]:
silence_count = 0

for genre in GENRES:
    genre_path = os.path.join(STEMS_PATH, genre)
    for song in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song)
        if os.path.isdir(song_path):
            for stem in os.listdir(song_path):
                if stem.endswith(".wav"):
                    file_path = os.path.join(song_path, stem)
                    y, sr = librosa.load(file_path, sr=None)
                    first_half = y[:int(0.5 * sr)]
                    if np.max(np.abs(first_half)) < 1e-4:
                        silence_count += 1

print("Files with silence in first 0.5 sec:", silence_count)


In [ ]:
y_pred = clf.predict(X_val)

macro_f1 = f1_score(y_val, y_pred, average='macro')

cm = confusion_matrix(y_val, y_pred)

cr = classification_report(y_val, y_pred)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=GENRES, yticklabels=GENRES)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

tp = {}
fp = {}
fn = {}
tn = {}

for i, genre in enumerate(GENRES):
    tp[genre] = cm[i, i]
    fp[genre] = cm[:, i].sum() - cm[i, i]
    fn[genre] = cm[i, :].sum() - cm[i, i]
    tn[genre] = cm.sum() - (tp[genre] + fp[genre] + fn[genre])

print("TP:", tp)
print("FP:", fp)
print("FN:", fn)
print("TN:", tn)

accuracy = np.trace(cm) / np.sum(cm)
print("Model Accuracy:", accuracy)
